# F-M-N-SUPP: Multifrequency Noisy Support Recovery

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root()
sys.path[:0] = [
    str(REPO_ROOT / "src"),
    str(REPO_ROOT / "notebooks" / "benchmarks_v2" / "functions"),
]

from figures import save_pdf
from multiple_frequency_config import (
    ALPHA,
    MAX_ITER,
    MIC_COUNTS,
    NUM_ACTIVE,
    NUM_SOURCES,
    SEEDS,
    SOURCE_ANGLE_SPAN,
    METHOD_COLORS,
    METHOD_LINESTYLES,
    base_sim_kwargs,
)
from single_frequency_benchmarks import plot_metric, run_snr_benchmark


def format_snr_axis(ax, results_df, ticks_to_label=None):
    snr_ticks = (
        results_df[["sensor_snr_db", "sensor_snr_label"]]
        .drop_duplicates()
        .sort_values("sensor_snr_db", ascending=False)
    )
    ax.set_xticks(snr_ticks["sensor_snr_db"])

    if ticks_to_label is None:
        ax.set_xticklabels(snr_ticks["sensor_snr_label"].str.replace(" dB", ""))
        ax.tick_params(axis="x", labelrotation=45)
    else:
        ax.set_xticklabels([
            label.replace(" dB", "") if label in ticks_to_label else ""
            for label in snr_ticks["sensor_snr_label"]
        ])
        ax.tick_params(axis="x", labelrotation=0)

    ax.invert_xaxis()


def plot_snr_metric(results_df, y_col, ylabel, title, save_name, ylim=None):
    fig, ax, summary_df = plot_metric(
        results_df,
        x_col="sensor_snr_db",
        y_col=y_col,
        method_order=METHOD_ORDER,
        colors=PLOT_COLORS,
        linestyles=PLOT_LINESTYLES,
        xlabel="Sensor SNR (dB)",
        ylabel=ylabel,
        title=title,
        xscale="log",
        yscale="linear",
    )
    format_snr_axis(ax, results_df)
    if ylim is not None:
        ax.set_ylim(*ylim)
    save_pdf(fig, FIGURE_DIR / save_name)
    plt.show()
    return summary_df


In [ ]:
TAG = "F-M-N-SUPP"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / "v2" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SNR_DB = [None, 60, 40, 30, 25, 20, 15, 13, 11, 9, 8, 7, 6, 5, 4, 3, 2, 1]
NUM_MICS = int(MIC_COUNTS[-1])
SEPARATION_DEG = np.rad2deg(SOURCE_ANGLE_SPAN)
SIM_KWARGS = base_sim_kwargs()
SIM_KWARGS["num_mics"] = NUM_MICS
SIM_KWARGS.pop("sensor_snr_db", None)

RIDGE_ALPHA = 1e-3
SPARSE_PRECISION = 1.0
SPARSE_EPS = 0.005

METHOD_ORDER = [
    "Sparse Prior, no groups from MP",
    "Sparse Prior, frequency groups from MP",
    "Sparse Prior, no groups from Ridge",
    "Sparse Prior, frequency groups from Ridge",
    "LASSO from MP",
    "LASSO from Ridge",
    "Moore-Penrose",
    "Ridge",
]
PLOT_COLORS = METHOD_COLORS | {
    "Sparse Prior, no groups from MP": "tab:red",
    "Sparse Prior, frequency groups from MP": "purple",
    "Sparse Prior, no groups from Ridge": "tab:red",
    "Sparse Prior, frequency groups from Ridge": "purple",
    "LASSO from MP": METHOD_COLORS["LASSO"],
    "LASSO from Ridge": METHOD_COLORS["LASSO"],
    "Moore-Penrose": "tab:blue",
    "Ridge": "#009E73",
}
PLOT_LINESTYLES = METHOD_LINESTYLES | {
    "Sparse Prior, no groups from MP": ":",
    "Sparse Prior, frequency groups from MP": "-",
    "Sparse Prior, no groups from Ridge": "--",
    "Sparse Prior, frequency groups from Ridge": "--",
    "LASSO from MP": "-",
    "LASSO from Ridge": "--",
    "Moore-Penrose": ":",
    "Ridge": ":",
}

RECOVERY_METHODS = {
    "Moore-Penrose": {
        "solver": "initializer",
        "initializer": "mp",
    },
    "Ridge": {
        "solver": "initializer",
        "initializer": "ridge",
        "ridge_alpha": RIDGE_ALPHA,
    },
    "LASSO from MP": {
        "solver": "lasso",
        "initializer": "mp",
        "lasso_alpha": ALPHA,
        "lasso_max_iter": MAX_ITER,
    },
    "LASSO from Ridge": {
        "solver": "lasso",
        "initializer": "ridge",
        "ridge_alpha": RIDGE_ALPHA,
        "lasso_alpha": ALPHA,
        "lasso_max_iter": MAX_ITER,
    },
    "Sparse Prior, no groups from MP": {
        "solver": "sparse_prior",
        "initializer": "mp",
        "sparse_grouping": "none",
        "sparse_precision": SPARSE_PRECISION,
        "sparse_eps": SPARSE_EPS,
    },
    "Sparse Prior, frequency groups from MP": {
        "solver": "sparse_prior",
        "initializer": "mp",
        "sparse_grouping": "frequency",
        "sparse_precision": SPARSE_PRECISION,
        "sparse_eps": SPARSE_EPS,
    },
    "Sparse Prior, no groups from Ridge": {
        "solver": "sparse_prior",
        "initializer": "ridge",
        "ridge_alpha": RIDGE_ALPHA,
        "sparse_grouping": "none",
        "sparse_precision": SPARSE_PRECISION,
        "sparse_eps": SPARSE_EPS,
    },
    "Sparse Prior, frequency groups from Ridge": {
        "solver": "sparse_prior",
        "initializer": "ridge",
        "ridge_alpha": RIDGE_ALPHA,
        "sparse_grouping": "frequency",
        "sparse_precision": SPARSE_PRECISION,
        "sparse_eps": SPARSE_EPS,
    },
}


In [ ]:
results_df = run_snr_benchmark(
    SNR_DB,
    SEEDS,
    SEPARATION_DEG,
    RECOVERY_METHODS,
    sim_kwargs=SIM_KWARGS,
)

results_df["method"] = pd.Categorical(
    results_df["method"], categories=METHOD_ORDER, ordered=True
)
results_df = results_df.sort_values(
    ["sensor_snr_db", "phase_seed", "method"]
).reset_index(drop=True)

# results_df.head()


In [ ]:
error_summary_df = plot_snr_metric(
    results_df,
    y_col="relative_complex_error",
    ylabel="Relative complex error",
    title=f"Multifrequency noisy recovery, {NUM_MICS} mics and {NUM_SOURCES} sources",
    save_name="relative_complex_error_vs_sensor_snr.pdf",
    ylim=(-0.05, 2.05)
)

error_summary_df.head()


In [ ]:
f1_summary_df = plot_snr_metric(
    results_df,
    y_col="f1_threshold_10_percent",
    ylabel=r"$F_1$-score",
    title=f"Multifrequency noisy support recovery, {NUM_MICS} mics and {NUM_SOURCES} sources",
    save_name="f1_threshold_10_percent_vs_sensor_snr.pdf",
    ylim=(-0.02, 1.02),
)

f1_summary_df.head()
